# <p style="font-family: Cambria; text-align: center; font-size: 48px;">  I. DATA PRE-PROCESSING</p>

<p style="font-family: Cambria; font-size: 20px; color:red;">
<b>Dataset Overview</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
This heart failure dataset consists of seven separate CSV files containing complementary information about hospitalized patients. 
The datasets cover demographic characteristics, cardiac complications and measurements, medical history, hospitalization and discharge information, laboratory investigations, patient responsiveness, and prescription records.
</i></b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
The common patient identifier across the datasets is 
<code>inpatient_number</code>. 
This identifier allows the patient-level tables to be connected during the data integration stage.
The prescription dataset has a different structure because one patient can have multiple prescription records.
</i></b>
</p>


<p style="font-family: Cambria; font-size: 18px; color:red;">
<b>Overview of Individual Datasets</b>
</p>

| Dataset | Description | Role in Analysis |
|---|---|---|
| **Demography** | Contains patient demographic characteristics such as gender, weight, height, BMI, occupation and age category. | Describes the patient's demographic and physical characteristics. |
| **Cardiac** | Contains cardiac measurements, echocardiographic information and cardiac complications. | Provides information about cardiac structure, function and severity. |
| **History** | Contains medical history, comorbidities and other background clinical conditions. | Represents the patient's underlying health conditions. |
| **Hospitalization** | Contains admission, hospitalization, discharge and outcome-related information. | Provides hospitalization details and clinical outcomes. |
| **Labs** | Contains laboratory and vital-sign measurements collected during hospitalization. | Provides biochemical and physiological measurements. |
| **Responsiveness** | Contains information related to the patient's responsiveness and neurological status. | Provides information about patient responsiveness and neurological condition. |
| **Prescriptions** | Contains medication/prescription records associated with patients. | Provides information about medications received by patients. |

<p style="font-family: Cambria; font-size: 16px;">
<b>Reasoning</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
Before cleaning the data, we need to understand how the information is organized.
Most tables are patient-level tables, while the prescription table has a one-to-many structure because a patient can receive multiple medications.
Understanding this structure prevents incorrect merging later in the analysis.
</i></b>
</p>

<p style="font-family: Cambria; font-size: 18px; color:red;">
<b>Dataset Structure</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
The six main clinical datasets are organized at the patient level, meaning that each patient is expected to have one record in each table.
The prescriptions dataset follows a one-to-many structure because a patient may have multiple medication records.
</i></b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
Because of this difference in structure, the prescription data should not be directly merged with the patient-level datasets.
Instead, prescription records will first be transformed into patient-level medication features before the final master dataset is created.
</i></b>
</p>

<p style="font-family: Cambria; font-size: 20px; color:red;">
<b>Heart Failure Dataset — Data Cleaning</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
The goal of this notebook is to clean and prepare the heart failure datasets for analysis and predictive modeling.
The workflow includes understanding the datasets, checking patient identifiers, correcting confirmed data-quality problems, handling missing values, standardizing variables, preparing prescription information, merging patient-level datasets, and performing final consistency checks.
</i></b>
</p>

In [1]:
#Importing all the Necessary Libraries
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from functools import reduce
import warnings
warnings.filterwarnings("ignore")


In [2]:
#Reading the files
dfdemography=pd.read_csv( "demography.csv")
dfhospitalization=pd.read_csv("hospitalization_discharge.csv")
dfcardiac=pd.read_csv("cardiac_complications.csv")
dfhistory=pd.read_csv("patienthistory.csv")
dfresponsiveness=pd.read_csv( "responsivenes.csv")
dflabs=pd.read_csv("labs.csv")
dfprescriptions=pd.read_csv("patient_precriptions.csv")

In [3]:
datasets = {
    "demography": dfdemography,
    "cardiac": dfcardiac,
    "hospitalization": dfhospitalization,
    "labs": dflabs,
    "prescriptions": dfprescriptions,
    "history": dfhistory,
    "responsiveness": dfresponsiveness
}
# Keep untouched copies
raw = {name: df.copy() for name, df in datasets.items()}

print("7 datasets loaded successfully.")

7 datasets loaded successfully.


<p style="font-family: Cambria; font-size: 20px; color:red;">
<b>Dataset Size</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b>Reasoning</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
Before making changes, we need to know how many rows and columns are present in each table. This provides a simple structural overview without using a broad data-quality audit.
</i></b>
</p>

In [4]:
for name, df in datasets.items():
    print(f"{name:20} | Rows: {len(df):5} | Columns: {df.shape[1]:3}")

demography           | Rows:  2009 | Columns:   7
cardiac              | Rows:  2008 | Columns:  14
hospitalization      | Rows:  2008 | Columns:  21
labs                 | Rows:  2008 | Columns: 107
prescriptions        | Rows: 15362 | Columns:   2
history              | Rows:  2008 | Columns:  17
responsiveness       | Rows:  2008 | Columns:   6


<p style="font-family: Cambria; font-size: 18px; color:red;">
<b>Step 1: Removing Wrong Patient Entry from Demography table</b>
</p>
<p style="font-family: Cambria; font-size: 16px;">
<b>Reasoning
</b>
<p style="font-family: Cambria; font-size: 16px;">
<b><i>The demography table has 2009 rows, but every other table has 2008. When we checked, the extra row was ID 5. 
    It has no gender, weight, height or age, but it still has a BMI of 46. BMI is calculated from weight and height,
so a BMI without them cannot be real. Also, all real patient IDs have 6 digits (like 722128), 
and ID 5 is not in any other table. So this is a wrong entry, not a real patient.</i></b>

In [5]:
dfdemography = dfdemography[dfdemography['inpatient_number'] != 5]
datasets['demography'] = dfdemography
dfdemography.shape

(2008, 7)

<p style="font-family: Cambria; font-size: 16px;">
<b>Insight</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
        After removing it, demography has 2008 rows, the same as the other tables. 
    Every patient in our final table is a real patient with hospital data,
so counts like "how many males and females" and the average BMI will be correct.
    </i></b>
</p>

<p style="font-family: Cambria; font-size: 18px; color:red;">
<b>Step 2: Setting Impossible Weight and Height to Null in Demography table</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b>Reasoning</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
Some patients have weight 0 kg or 8 kg, and height 0.35 m or 0.48 m. All patients in this data are adults (youngest age group is 21-29), and an adult cannot weigh 0 kg or be 35 cm tall. These are typing mistakes or "not recorded" values. We changed weight below 20 kg and height below 1 m to null instead of guessing a value, because guessing someone's weight would be making up data.
</i></b>
</p>

In [6]:
dfdemography.loc[dfdemography['weight'] < 20, 'weight'] = np.nan
dfdemography.loc[dfdemography['height'] < 1, 'height'] = np.nan

<p style="font-family: Cambria; font-size: 16px;">
<b>Insight</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
Because of these values, some BMIs were 0 and 404, while a normal human BMI is between about 15 and 40. Questions like "Do underweight patients have higher death rates?" depend on correct BMI, and one BMI of 404 alone can push the average up a lot.
</i></b>
</p>

<p style="font-family: Cambria; font-size: 18px; color:red;">
<b>Step 3: Recalculating BMI in Demography table</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b>Reasoning</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
After fixing the wrong weight and height in Step 2, the old BMI column still had the wrong values (0 and 404). BMI should always match the weight and height of the same patient, so we recalculated BMI = weight / height² for all patients.
</i></b>
</p>

In [7]:
dfdemography['bmi'] = round(dfdemography['weight'] / (dfdemography['height'] ** 2), 2)
dfdemography['bmi'].describe()

count    2000.000000
mean       21.296960
std         3.807773
min        13.060000
25%        18.490000
50%        20.760000
75%        23.440000
max        39.110000
Name: bmi, dtype: float64

<p style="font-family: Cambria; font-size: 16px;">
<b>Insight</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
After the fix, BMI is between 13 and 39 with an average of about 21, which makes sense for real patients. 2000 patients have a valid BMI. BMI can now be used safely in descriptive analysis and in the prediction model.
</i></b>
</p>

<p style="font-family: Cambria; font-size: 18px; color:red;">
<b>Step 4: Filling Missing Occupation and Fixing Text Case in Demography table</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b>Reasoning</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
27 patients had no occupation. Also, 'farmer' and 'worker' were written in lowercase while 'Officer' and 'UrbanResident' start with capitals. We used 'Unknown' because we truly don't know their job. The hospitalization table already uses 'Unknown' in the same way for discharge destination.
</i></b>
</p>

In [8]:
dfdemography['occupation'] = dfdemography['occupation'].fillna('Unknown')
dfdemography['occupation'] = dfdemography['occupation'].replace({'farmer': 'Farmer', 'worker': 'Worker'})
dfdemography['occupation'].value_counts()

occupation
UrbanResident    1670
Farmer            198
Others             89
Unknown            27
Worker             17
Officer             7
Name: count, dtype: int64

<p style="font-family: Cambria; font-size: 16px;">
<b>Insight</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
Most patients are urban residents (1670 of 2008), followed by farmers (198). When we group patients by occupation, pandas leaves out null rows by default, so these 27 patients would have disappeared from charts. With 'Unknown' they stay in the counts, and same-style labels make charts look clean.
</i></b>
</p>

<p style="font-family: Cambria; font-size: 18px; color:red;">
<b>Step 5: Filling Blank Respiratory Support in Hospitalization table</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b>Reasoning</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
Only 42 patients have a value here: IMV (breathing machine with a tube) or NIMV (breathing mask). The other 1966 are blank. Ventilation is a big treatment and the hospital would always record it if it was given. So blank means "no ventilation was given", not "we don't know".
</i></b>
</p>

In [9]:
#Blank respiratory support then no ventilation
dfhospitalization['respiratory_support'] = dfhospitalization['respiratory_support'].fillna('None')
dfhospitalization['respiratory_support'].value_counts()

respiratory_support
None    1966
IMV       25
NIMV      17
Name: count, dtype: int64

<p style="font-family: Cambria; font-size: 16px;">
<b>Insight</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
Only about 2% of patients needed breathing support (25 IMV, 17 NIMV). Without the fix, grouping by this column counts only 42 of 2008 patients. Now we can properly compare, for example, the death rate of ventilated vs non-ventilated patients.
</i></b>
</p>

<p style="font-family: Cambria; font-size: 18px; color:red;">
<b>Step 6: Converting Admission Date to Date Format in Hospitalization table</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b>Reasoning</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
The admission date was stored as text, so Python cannot use it as a date. After converting, we can take out the year or month and sort by time.
</i></b>
</p>

In [10]:
#Convert admission date from text to date
dfhospitalization['admission_date'] = pd.to_datetime(dfhospitalization['admission_date'])
dfhospitalization['admission_date'].dtype

dtype('<M8[ns]')

<p style="font-family: Cambria; font-size: 16px;">
<b>Insight</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
The column is now a proper date type. We can answer questions like "Did admissions increase from 2016 to 2019?" or "Are more patients admitted in winter?"
</i></b>
</p>

<p style="font-family: Cambria; font-size: 18px; color:red;">
<b>Step 7: Keeping Nulls in Time of Death and Readmission Time in Hospitalization table</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b>Reasoning</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
These columns are blank for most patients, and that is correct. Blank in time_of_death means the patient did not die, and blank in readmission_time means the patient did not come back. Filling with 0 would mean the patient died or was readmitted on day 0, which is wrong. So no change was made.
</i></b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b>Insight</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
Not every null is a problem; here the blank itself carries meaning. Averages like "average days to readmission" will use only patients who were really readmitted.
</i></b>
</p>

<p style="font-family: Cambria; font-size: 18px; color:red;">
<b>Step 8: Renaming LV Diameter to mm and Removing Wrong Values in Cardiac table</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b>Reasoning</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
This column measures the size of the heart's main pumping chamber. The data dictionary says the unit is cm (normal 3.5 to 5.6 cm), but most values are around 53. That only makes sense in mm (53 mm = 5.3 cm, a normal size). So the data is in mm, and we renamed the column to lvedd_mm so nobody compares it with the cm range by mistake. Two values (0.3 and 0.93) are too small in both mm and cm, so they were set to null.
</i></b>
</p>

In [11]:
#Rename column to show correct unit (mm) and remove impossible values
dfcardiac.rename(columns={'left_ventricular_end_diastolic_diameter_lv': 'lvedd_mm'}, inplace=True)
dfcardiac.loc[dfcardiac['lvedd_mm'] < 10, 'lvedd_mm'] = np.nan
dfcardiac['lvedd_mm'].describe()

count    1309.000000
mean       53.190222
std        10.738113
min        22.000000
25%        45.000000
50%        53.000000
75%        60.000000
max        88.000000
Name: lvedd_mm, dtype: float64

<p style="font-family: Cambria; font-size: 16px;">
<b>Insight</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
After the fix, values range from 22 to 88 mm with an average of 53 mm. A larger heart chamber means worse heart failure, so this is an important column. With the right unit we can correctly find patients with an enlarged heart.
</i></b>
</p>

<p style="font-family: Cambria; font-size: 18px; color:red;">
<b>Step 9: Converting Mitral Valve E and A Values to m/s in Cardiac table</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b>Reasoning</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
These values show how fast blood flows into the heart and are used to check if the heart relaxes properly. Normal values are around 0.3 to 2 m/s. A few values were between 10 and 100, which means they were typed in cm/s. For eg: one patient has E = 44, A = 99 and E/A ratio = 0.44. That ratio only works if E = 0.44 and A = 0.99 m/s. So we divided these values by 100. Values from 401 to 409 don't make sense in either unit, so they were set to null.
</i></b>
</p>

In [12]:
#Remove values above 100, then change cm/s values to m/s
for col in ['mitral_valve_ems', 'mitral_valve_ams']:
    dfcardiac.loc[dfcardiac[col] > 100, col] = np.nan
    dfcardiac.loc[dfcardiac[col] >= 10, col] = dfcardiac[col] / 100

dfcardiac[['mitral_valve_ems', 'mitral_valve_ams']].describe()

,mitral_valve_ems,mitral_valve_ams
count,971.000000,546.000000
mean,1.059114,0.832410
std,0.459453,0.317554
min,0.030000,0.060000
25%,0.760000,0.580000
50%,1.000000,0.820000
75%,1.260000,1.040000
max,3.900000,2.800000


<p style="font-family: Cambria; font-size: 16px;">
<b>Insight</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
Before the fix the average E value was 4.85, which is impossible. After the fix the average E is about 1.06 m/s and A is about 0.83 m/s, which is normal.
</i></b>
</p>

<p style="font-family: Cambria; font-size: 18px; color:red;">
<b>Step 10: Removing Very Low Tricuspid Valve Pressure from Cardiac table</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b>Reasoning</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
This value estimates the pressure in the lung arteries. It is almost never below 10 mmHg, and in heart failure it is usually higher. Values like 1.7 and 2.1 look like numbers from the velocity column typed into the pressure column by mistake, so values below 10 were set to null.
</i></b>
</p>

In [13]:
#Remove impossible low pressure values
dfcardiac.loc[dfcardiac['tricuspid_valve_return_pressure'] < 10, 'tricuspid_valve_return_pressure'] = np.nan

<p style="font-family: Cambria; font-size: 16px;">
<b>Insight</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
High lung pressure is a warning sign in heart failure. Wrong low values would have made some sick patients look normal.
</i></b>
</p>

<p style="font-family: Cambria; font-size: 18px; color:red;">
<b>Step 11: Calculating Missing E/A Ratio in Cardiac table</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b>Reasoning</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
E/A is simply E divided by A. For 143 patients, both E and A were recorded but the ratio was empty. Since we know both numbers, this is the exact value and not a guess.
</i></b>
</p>

In [14]:
#Calculate E/A where E and A are available but ratio is missing
missing_ea = dfcardiac['ea'].isnull() & dfcardiac['mitral_valve_ems'].notnull() & dfcardiac['mitral_valve_ams'].notnull()
dfcardiac.loc[missing_ea, 'ea'] = round(dfcardiac['mitral_valve_ems'] / dfcardiac['mitral_valve_ams'], 3)
print("E/A ratio filled for:", missing_ea.sum(), "patients")

E/A ratio filled for: 143 patients


<p style="font-family: Cambria; font-size: 16px;">
<b>Insight</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
We now have the E/A ratio for 536 patients instead of 393, about 36% more patients. More patients means stronger results.
</i></b>
</p>

<p style="font-family: Cambria; font-size: 18px; color:red;">
<b>Step 12: Checking the Leukemia Column in Patient History table</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b>Reasoning</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
Leukemia is 0 (no) for all 2008 patients. A column where everyone has the same value cannot show any difference between patients. We kept the column so the table stays as given, but we will leave it out of correlation and the prediction model.
</i></b>
</p>

In [15]:
#All patients have leukemia = 0, so the column has no information
dfhistory['leukemia'].value_counts()

leukemia
0    2008
Name: count, dtype: int64

<p style="font-family: Cambria; font-size: 16px;">
<b>Insight</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
No patient in this data has leukemia, so it cannot explain death or readmission. In a correlation matrix this column would show as empty (NaN), so it should be skipped there.
</i></b>
</p>

<p style="font-family: Cambria; font-size: 18px; color:red;">
<b>Step 13: Converting Type II Respiratory Failure to 1 and 0 in Patient History table</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b>Reasoning</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
All other disease columns in this table use 1 = yes and 0 = no, but this one had the words 'typeii' and 'nontypeii'. We changed it to 1 and 0 so all disease columns follow the same rule.
</i></b>
</p>

In [16]:
#Change text to 1 and 0 like other disease columns
dfhistory['type_ii_respiratory_failure'] = dfhistory['type_ii_respiratory_failure'].map({'typeii': 1, 'nontypeii': 0})
dfhistory['type_ii_respiratory_failure'].value_counts()

type_ii_respiratory_failure
0    1894
1     114
Name: count, dtype: int64

<p style="font-family: Cambria; font-size: 16px;">
<b>Insight</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
114 patients (about 6%) have type II respiratory failure. With 1 and 0, the average of the column gives the % of patients with the disease, and it can be used in correlation and in the prediction model, which skip text columns.
</i></b>
</p>

<p style="font-family: Cambria; font-size: 18px; color:red;">
<b>Step 14: Keeping the Few Nulls in Disease Columns in Patient History table</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b>Reasoning</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
Only 1 to 5 patients out of 2008 have nulls in peptic ulcer, kidney disease, liver disease and CCI score. Filling them with 0 would mean saying "this patient does not have kidney disease", which we don't know. So we did not fill them.
</i></b>
</p>

In [17]:
dfhistory.isnull().sum()

inpatient_number                             0
cerebrovascular_disease                      0
dementia                                     0
chronic_obstructive_pulmonary_disease        0
connective_tissue_disease                    0
peptic_ulcer_disease                         2
diabetes                                     0
moderate_to_severe_chronic_kidney_disease    2
hemiplegia                                   0
leukemia                                     0
malignant_lymphoma                           0
solid_tumor                                  0
liver_disease                                1
aids                                         0
cci_score                                    5
type_ii_respiratory_failure                  0
acute_renal_failure                          0
dtype: int64

<p style="font-family: Cambria; font-size: 16px;">
<b>Insight</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
Such a small number (less than 0.3% of patients) will not change our results, and pandas skips nulls when calculating averages. We don't add false information about any patient's health.
</i></b>
</p>

<p style="font-family: Cambria; font-size: 18px; color:red;">
<b>Step 15: Checking GCS Score in Responsiveness table</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b>Reasoning</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
GCS (Glasgow Coma Scale) measures how awake a patient is. It is the sum of eye opening + verbal response + movement. We checked this sum for every patient. There are no nulls either, so no cleaning was needed.
</i></b>
</p>

In [18]:
#Check GCS = eye + verbal + movement (0 means all rows are correct)
(dfresponsiveness['eye_opening'] + dfresponsiveness['verbal_response'] + dfresponsiveness['movement'] != dfresponsiveness['gcs']).sum()

np.int64(0)

<p style="font-family: Cambria; font-size: 16px;">
<b>Insight</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
The sum matched for all 2008 patients (result 0 below), so we can trust this table as it is. Checking and finding no problem is also part of data cleaning.
</i></b>
</p>

<p style="font-family: Cambria; font-size: 18px; color:red;">
<b>Step 16: Checking Cholinesterase and Blood Gas Temperature in Labs table</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b>Reasoning</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
cholinesterase is empty for all patients, so there is nothing to analyse. body_temperature_blood_gas is 37 for every patient, which is the default setting of the blood gas machine, not a real temperature. Real temperatures are in the body_temperature column. We kept both columns so the table stays as given, but we will not use them in analysis.
</i></b>
</p>

In [19]:
print("cholinesterase non-null values:", dflabs['cholinesterase'].notnull().sum())
print("body_temperature_blood_gas values:", dflabs['body_temperature_blood_gas'].unique())

cholinesterase non-null values: 0
body_temperature_blood_gas values: [37. nan]


<p style="font-family: Cambria; font-size: 16px;">
<b>Insight</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
These two columns give no information about patients. Anyone reading the data should use body_temperature for real temperature, not the blood gas column.
</i></b>
</p>

<p style="font-family: Cambria; font-size: 18px; color:red;">
<b>Step 17: Renaming Wrongly Named Columns in Labs table</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b>Reasoning</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
Some column names look like bad translations. For eg: glutamic_oxaliplatin sounds like a cancer drug, but the data dictionary says it is the AST/ALT ratio (a liver test). high_sensitivity_protein is actually hs-CRP, an inflammation marker. We used the data dictionary to give them the correct short names (ast, alt, ast_alt_ratio, albumin_globulin_ratio, hs_crp).
</i></b>
</p>

In [20]:
#Rename columns using the data dictionary
dflabs.rename(columns={'glutamic_oxaloacetic_transaminase': 'ast',
                       'glutamic_pyruvic_transaminase': 'alt',
                       'glutamic_oxaliplatin': 'ast_alt_ratio',
                       'white_globulin_ratio': 'albumin_globulin_ratio',
                       'high_sensitivity_protein': 'hs_crp'}, inplace=True)

<p style="font-family: Cambria; font-size: 16px;">
<b>Insight</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
When we pick columns for analysis, we now know what each one really means, and the names are short and easy to use in code and charts.
</i></b>
</p>

<p style="font-family: Cambria; font-size: 18px; color:red;">
<b>Step 18: Removing Zero Pulse, Respiration and Blood Pressure from Labs table</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b>Reasoning</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
A patient who is alive at admission cannot have a pulse of 0 or blood pressure of 0. The 0 was entered because the value was not recorded. One patient had 0 for all five readings.
</i></b>
</p>

In [21]:
vitals = ['pulse', 'respiration', 'systolic_blood_pressure', 'diastolic_blood_pressure', 'map_value']

#Count of 0 values before fix
(dflabs[vitals] == 0).sum()

pulse                       1
respiration                 1
systolic_blood_pressure     3
diastolic_blood_pressure    3
map_value                   3
dtype: int64

In [22]:
dflabs[vitals] = dflabs[vitals].replace(0, np.nan)

In [23]:
#Count of 0 values after fix
(dflabs[vitals] == 0).sum()

pulse                       0
respiration                 0
systolic_blood_pressure     0
diastolic_blood_pressure    0
map_value                   0
dtype: int64

<p style="font-family: Cambria; font-size: 16px;">
<b>Insight</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
There were only a few such values (1 to 3 per column), and after the fix there are none. Low blood pressure in heart failure is linked to higher death risk. If the 0s stayed, these patients would look like they had extremely low BP, and the average BP would come out lower than it really is.
</i></b>
</p>

<p style="font-family: Cambria; font-size: 18px; color:red;">
<b>Step 19: Removing Blood Pressure Where Systolic Is Lower Than Diastolic from Labs table</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b>Reasoning</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
Systolic BP (top number) is always higher than diastolic BP (bottom number); that is how blood pressure works. For 2 patients it was the other way, so the numbers were typed wrong. We can't know the right values, so we set them to null. MAP was also cleared because it is calculated from these two values.
</i></b>
</p>

In [24]:
#Systolic lower than diastolic is not possible
wrong_bp = dflabs['systolic_blood_pressure'] < dflabs['diastolic_blood_pressure']
print("Wrong BP rows:", wrong_bp.sum())
dflabs.loc[wrong_bp, ['systolic_blood_pressure', 'diastolic_blood_pressure', 'map_value']] = np.nan

Wrong BP rows: 2


<p style="font-family: Cambria; font-size: 16px;">
<b>Insight</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
Only 2 rows were affected, so the impact is small, but now only real blood pressure readings are used in analysis.
</i></b>
</p>

<p style="font-family: Cambria; font-size: 18px; color:red;">
<b>Step 20: Fixing Hematocrit and Troponin Units in Labs table</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b>Reasoning</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
Hematocrit was stored as a fraction (like 0.36), while the dictionary and the second hematocrit column use % (like 36). We multiplied by 100 so both use the same unit. Troponin (heart damage marker) had a median of 0.055, which matches ng/mL, but the dictionary uses pg/mL with normal limit 14. So we multiplied by 1000.
</i></b>
</p>

In [25]:
#Hematocrit fraction to %, troponin ng/mL to pg/mL
dflabs['hematocrit'] = dflabs['hematocrit'] * 100
dflabs['high_sensitivity_troponin'] = dflabs['high_sensitivity_troponin'] * 1000

print("Hematocrit median:", dflabs['hematocrit'].median())
print("Troponin median:", dflabs['high_sensitivity_troponin'].median())

Hematocrit median: 35.9
Troponin median: 55.0


<p style="font-family: Cambria; font-size: 16px;">
<b>Insight</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
In the old unit only 0.3% of patients would show high troponin, which is wrong because heart failure patients very often have high troponin. After the fix, 80% show high troponin, which is what we expect. Now the median hematocrit is 35.9% and median troponin is 55 pg/mL, and questions like "how many patients have heart damage?" get the correct answer.
</i></b>
</p>

<p style="font-family: Cambria; font-size: 18px; color:red;">
<b>Step 21: Keeping Missing Lab Values Unfilled in Labs table</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b>Reasoning</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
Some tests were not done for every patient. For eg: blood gas is missing for about 50% of patients. The missing values are not random, because patients who had the blood gas test were sicker (more NYHA class IV and nearly 3 times the 28-day death rate). If we fill gaps with the average, we give healthier untested patients the values of sicker tested patients, and the data would not show the real picture.
</i></b>
</p>

In [26]:
#% missing for lab columns (top 15)
round(dflabs.isnull().mean() * 100, 1).sort_values(ascending=False).head(15)

cholinesterase                    100.0
homocysteine                       92.7
apolipoprotein_a                   91.2
apolipoprotein_b                   91.2
lipoprotein                        91.2
erythrocyte_sedimentation_rate     84.7
myoglobin                          80.2
inorganic_phosphorus               79.7
serum_magnesium                    79.7
ast_alt_ratio                      70.5
hs_crp                             53.1
reduced_hemoglobin                 50.6
methemoglobin                      50.6
hematocrit_blood_gas               50.6
total_hemoglobin                   50.6
dtype: float64

<p style="font-family: Cambria; font-size: 16px;">
<b>Insight</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
Some tests like homocysteine and apolipoprotein are missing for more than 90% of patients, so they are not useful for analysis. Every analysis uses only real test results. If the prediction model needs filled values, we will do it in the model notebook after splitting training and test data.
</i></b>
</p>

<p style="font-family: Cambria; font-size: 18px; color:red;">
<b>Step 22: Adding a Flag for BNP at Machine Limit in Labs table</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b>Reasoning</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
BNP is the main blood marker for heart failure. 100 patients have BNP exactly 5000, and the next highest value is 4982. 5000 is the highest number the lab machine can report, so the real BNP for these patients could be much higher. We kept these values because these are the sickest patients and they are important. We added a column bnp_at_limit (1 = yes) to show this.
</i></b>
</p>

In [27]:
#Flag BNP values at machine limit
dflabs['bnp_at_limit'] = (dflabs['brain_natriuretic_peptide'] == 5000).astype(int)
dflabs['bnp_at_limit'].value_counts()

bnp_at_limit
0    1908
1     100
Name: count, dtype: int64

<p style="font-family: Cambria; font-size: 16px;">
<b>Insight</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
About 5% of patients (100 of 2008) are at the limit. When we report average BNP, we know it is lower than the real value, and we can compare this high BNP group separately.
</i></b>
</p>

<p style="font-family: Cambria; font-size: 18px; color:red;">
<b>Step 23: Changing Prescriptions to One Row per Patient in Prescriptions table</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b>Reasoning</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
In this table each drug given to a patient is a separate row, so one patient appears about 8 times (up to 16). If we merge it directly, each patient would be copied 8 times in the final table, and every count and average would be wrong. For eg: one patient who died would be counted as 8 deaths. We used crosstab so each drug becomes a column with 1 (given) or 0 (not given), with one row per patient. 
   <p style="font-family: Cambria; font-size: 16px; color:blue;"> Feature Engineering :</p> We also added total_drugs, the number of different drugs each patient got.
</i></b>
</p>

In [28]:
#Rows per patient before pivot
dfprescriptions.groupby('inpatient_number').size().describe()

count    2007.000000
mean        7.654210
std         2.426591
min         1.000000
25%         6.000000
50%         8.000000
75%         9.000000
max        16.000000
dtype: float64

In [29]:
#Pivot: one row per patient, one column per drug
dfprescriptions_pivot = pd.crosstab(dfprescriptions['inpatient_number'], dfprescriptions['drug_name'])
dfprescriptions_pivot['total_drugs'] = dfprescriptions_pivot.sum(axis=1)
dfprescriptions_pivot = dfprescriptions_pivot.reset_index()
dfprescriptions_pivot.shape

(2007, 27)

In [30]:
#Show all columns of the pivoted table
pd.set_option('display.max_columns', None)
dfprescriptions_pivot.head()

drug_name,inpatient_number,Aspirin enteric-coated tablet,Atorvastatin calcium tablet,Benazepril hydrochloride tablet,Clopidogrel Hydrogen Sulphate tablet,Deslanoside injection,Digoxin tablet,Dobutamine hydrochloride injection,Enoxaparin Sodium injection,Furosemide injection,Furosemide tablet,Heparin Sodium injection,Hydrochlorothiazide tablet,Isoprenaline Hydrochloride injection,Isosorbide Mononitrate Sustained Release tablet,Meglumine Adenosine Cyclophosphate for injection,Metoprolol Succinate Sustained-release tablet,Milrinone injection,Nitroglycerin injection,Shenfu injection,Spironolactone tablet,Torasemide tablet,Valsartan Dispersible tablet,metoprolol tartrate injection,sulfotanshinone sodium injection,warfarin sodium tablet,total_drugs
0,722128,0,0,0,0,1,1,0,0,1,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,7
1,723327,1,1,0,1,1,1,0,0,1,1,0,0,0,1,0,0,1,0,0,1,1,1,0,0,0,12
2,723617,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,4
3,724385,0,0,0,0,1,1,0,0,1,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,6
4,725509,0,0,0,0,1,1,0,0,1,1,0,0,0,0,0,0,1,0,1,1,1,0,1,0,0,9


In [31]:
#How many patients got each drug
dfprescriptions_pivot.drop(columns=['inpatient_number', 'total_drugs']).sum().sort_values(ascending=False)

drug_name
Spironolactone tablet                               1833
Furosemide injection                                1718
Furosemide tablet                                   1641
Meglumine Adenosine Cyclophosphate for injection    1114
Deslanoside injection                               1015
Digoxin tablet                                       998
Aspirin enteric-coated tablet                        958
Atorvastatin calcium tablet                          822
Milrinone injection                                  707
sulfotanshinone sodium injection                     570
Metoprolol Succinate Sustained-release tablet        523
Benazepril hydrochloride tablet                      434
Clopidogrel Hydrogen Sulphate tablet                 401
Valsartan Dispersible tablet                         348
Shenfu injection                                     338
Isosorbide Mononitrate Sustained Release tablet      326
metoprolol tartrate injection                        307
Hydrochlorothiazide t

<p style="font-family: Cambria; font-size: 16px;">
<b>Insight</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
The most common drugs are Spironolactone (1833 patients) and Furosemide (injection 1718, tablet 1641), which are standard heart failure medicines. Now we can answer questions like "Do patients on Digoxin get readmitted more?", with each patient counted only once.
</i></b>
</p>

<p style="font-family: Cambria; font-size: 18px; color:red;">
<b>Step 24: Merging All Tables into One Final Table</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b>Reasoning</b>
</p>

<p style="font-family: Cambria; font-size: 16px;">
<b><i>
All 7 tables have the same patient ID, so we joined them into one table where each row is one patient with all their details. We started from the hospitalization table because it has the outcome columns (death and readmission) that most of our questions are about. We used a left merge so no patient gets dropped. One patient had no prescriptions, so their drug columns were filled with 0, meaning "not given".
</i></b>
</p>

In [32]:
#Merge all tables on inpatient_number
dffinal = dfhospitalization.merge(dfdemography, on='inpatient_number', how='left')
dffinal = dffinal.merge(dfcardiac, on='inpatient_number', how='left')
dffinal = dffinal.merge(dfhistory, on='inpatient_number', how='left')
dffinal = dffinal.merge(dfresponsiveness, on='inpatient_number', how='left')
dffinal = dffinal.merge(dflabs, on='inpatient_number', how='left')
dffinal = dffinal.merge(dfprescriptions_pivot, on='inpatient_number', how='left')

#Patient with no prescriptions gets 0 in drug columns
drug_cols = list(dfprescriptions_pivot.columns.drop('inpatient_number'))
dffinal[drug_cols] = dffinal[drug_cols].fillna(0)

dffinal.shape

(2008, 194)